# Lifecycle Lab：把 mini-sglang 的 API 与 tokenizer 留白接进请求链

本节补上[毕业表](../README.md)第 1 项，同时收录源码里的 API/tokenizer TODO 与 detokenizer NOTE。原有五个 lab 不变。本节先做阅读与可观察性实验，再任选一个 TODO 设计可验收的改动；不把源码标记误写成已完成的功能。

> **运行位置**：阅读和静态检查可在本地完成；启动模型与请求验证需 Linux + NVIDIA CUDA 服务器、可加载模型。


## 任务标签与转场规则

- **【必做】**：在 mini 完成调用链、真实轨迹或对拍；这些组成转去 SGLang 主仓库的门槛。
- **【小改选做】**：改动范围局部，适合加深理解；做一两项即可，不必全部完成。
- **【大改理论】**：保留原题的源码入口与验收设想，只交边界分析、反例和可复现测试计划；完整实现应在确认上游需求后进入 SGLang 主仓库。

完成本 notebook 的【必做】后，继续做 [lab 首页](../README.md) 所列其余章节的【必做】；全部完成便转去 SGLang 主仓库选题，无须等选做和大改题的完整实现。


## 【必做】先画一条请求的双向链

从 [`api_server.py`](../../python/minisgl/server/api_server.py) 的 `/generate`、`/v1/chat/completions` 或 shell completion 任选一个入口，沿 `TokenizeMsg` → [`TokenizeManager.tokenize`](../../python/minisgl/tokenizer/tokenize.py) → scheduler → `DetokenizeMsg` → [`DetokenizeManager.detokenize`](../../python/minisgl/tokenizer/detokenize.py) → `UserReply` → SSE/JSON 输出画图。标出 uid、prompt 类型、input ids、sampling 参数、增量文本和 finished 标记分别在哪里生成或改变。

**验收**：同一短请求记录服务端入口、scheduler 的 batch 与客户端收到的 chunk 顺序；流式结束标记只能在最终内容之后。断开流式客户端后，沿 `abort_user` 追踪回收路径。可复用 [Scheduler Lab](../scheduler/scheduler.ipynb) 的 trace 思路。


## 【必做】源码原有 TODO：参数映射与批量 tokenization

| 标签 | 源码标记 | 当前边界 | 动手题与验收 |
|---|---|---|---|
| 【大改理论】 | [`v1_completions` 的 `TODO: support more sampling parameters`](../../python/minisgl/server/api_server.py) | `OpenAICompletionRequest` 有比送进 `SamplingParams` 更多的字段。 | 逐字段列出“请求可接受 → 内部是否传递 → sampler 是否消费”的表。先用 `n=2` 与默认值对比返回的 choices 数量，作为“接口接受但尚未兑现”基线；再任选一个未兑现字段，写出它从请求到采样/终止/输出所需的最小契约和行为验收。若实际实现，必须用输出验证，不能只检查 HTTP 200。 |
| 【大改理论】 | [`shell_completion` 的同一 TODO](../../python/minisgl/server/api_server.py) | shell 路径单独构造 `SamplingParams`。 | 对照 chat 路径检查所选参数是否一致传递；记录一个未兑现字段在 shell 路径的实际行为。不要承诺内部尚无语义的参数已经生效。 |
| 【小改选做】 | [`TokenizeManager.tokenize` 的 `TODO: batch tokenization`](../../python/minisgl/tokenizer/tokenize.py) | 当前逐条处理 `msgs`；chat 消息先套 template，普通字符串直接编码。 | 给含不同长度的普通 prompt 与 chat prompt 建立逐条基线；若尝试批处理，比较每条未填充的 token ids、顺序、dtype 与空 batch 行为，并测量吞吐。 |

【必做】先完成字段映射表与逐条 tokenization 基线。上表中的实现按标签处理；这两个方向可独立完成。改动前先保存相同输入的输出；改动后只在对应路径上比较，不把吞吐提升或参数生效凭想象写进结论。


## 【必做】源码原有 NOTE：增量文本的 Unicode 边界

[`detokenize.py` 的 CJK `Note`](../../python/minisgl/tokenizer/detokenize.py) 提醒：CJK Unicode 区块并不覆盖所有日文、韩文字符。这段辅助逻辑借自上游实现；它影响 `find_printable_text` 何时释放尚未完整的词，不代表所有东亚文字都应逐 token 立刻输出。

**观察题**：选中文、日文假名、韩文谚文和英文各一个会跨多个 token 的短输入，记录每次 `DetokenizeMsg` 后的增量字符串和最终拼接文本；解释哪些中间步骤会被暂存、何时释放。最终拼接文本与一次性 decode 对比，同时检查不会重复或漏字。不要用“字形看起来相似”代替 Unicode code point 分类。

**交付**：请求链图、字段表、每条 TODO/NOTE 对应的观测结果，以及 TODO 的现状证据；小改选做如实际完成，再附失败/通过对照。
